# CS164 Optimization Methods — Session 17: Duality

**Source and provenance:** Reconstructed from the submitted ten-page *CS164 Session 17.pdf*, rather than an original workbook export. The PDF includes Questions 1–4 and four Python code cells; it ends at the opening of Question 5. The answers to Question 5 and Question 6 are not visible and have **not** been invented.

**Main topics:** primal versus dual optimization, graphical feasible sets, CVXPY verification, Lagrangian penalties, the infinite-step penalty, and the duality gap.

**Running this notebook:** Use a Python 3 kernel from the repository root or notebooks/ directory. Install the packages in requirements.txt. The companion standalone implementation is ../src/duality.py; it can also be run directly with:

    python src/duality.py
    python src/duality.py --save-dir session17_figures

The notebook imports reusable plotting/solver functions from the script to keep both representations consistent.

## Q1 — Concrete mixture: primal and dual

We buy two types of concrete. Type 1 costs $5 per lb and contains 30% cement, 40% gravel, and 30% sand; type 2 costs $1 per lb and contains 10% cement, 20% gravel, and 70% sand. We need at least 5 lb of cement, 3 lb of gravel, and 4 lb of sand.

Let x1 and x2 be the pounds purchased of types 1 and 2. The **primal** minimizes purchasing cost:

$$
\begin{aligned}
\min_{x_1,x_2\ge 0}\quad &5x_1+x_2\\
\text{s.t.}\quad &0.3x_1+0.1x_2\ge5 &&\text{(cement)}\\
&0.4x_1+0.2x_2\ge3 &&\text{(gravel)}\\
&0.3x_1+0.7x_2\ge4 &&\text{(sand)}.
\end{aligned}
$$

In matrix notation, minimize $c^Tx$ subject to $Ax\ge b$, $x\ge0$, with

$$
A=\begin{bmatrix}0.3&0.1\\0.4&0.2\\0.3&0.7\end{bmatrix},\qquad
b=\begin{bmatrix}5\\3\\4\end{bmatrix},\qquad
c=\begin{bmatrix}5\\1\end{bmatrix}.
$$

Write the constraints as $b-Ax\le0$ and introduce multipliers $\mu\ge0$:

$$
\mathcal L(x,\mu)=c^Tx+\mu^T(b-Ax)
=(c-A^T\mu)^Tx+b^T\mu.
$$

The infimum over $x\ge0$ is finite only if $A^T\mu\le c$. Therefore the **dual** is:

$$
\begin{aligned}
\max_{\mu_1,\mu_2,\mu_3\ge0}\quad &5\mu_1+3\mu_2+4\mu_3\\
\text{s.t.}\quad &0.3\mu_1+0.4\mu_2+0.3\mu_3\le5,\\
&0.1\mu_1+0.2\mu_2+0.7\mu_3\le1.
\end{aligned}
$$

The submitted graphical solution marks **primal x* = (0, 50)** and **dual mu* = (10, 0, 0)**. Their optimal objective values are both $50, although their feasible regions are different: the primal is an unbounded region in 2D, and the dual is a bounded tetrahedron in 3D.

In [ ]:
# Import the companion Python script; supports launch from the repo root
# or the notebooks/ directory.
from pathlib import Path
import sys

root = Path.cwd()
if not (root / "src" / "duality.py").exists():
    root = root.parent
if not (root / "src" / "duality.py").exists():
    raise FileNotFoundError("Open from the CS164 repository or notebooks/ directory")
sys.path.insert(0, str(root))

import numpy as np
import matplotlib.pyplot as plt
from src.duality import (
    A, B, COST, solve_concrete_primal_dual, plot_concrete_feasible_sets,
    lagrangian, plot_lagrangian_penalties, finite_step, plot_finite_step,
)

print("Material composition matrix:\n", A)
print("Minimum requirements:", B)
print("Costs per pound:", COST)

In [ ]:
# Q1 — Plot both feasible sets, using a 2D primal and 3D dual view.
fig = plot_concrete_feasible_sets()
plt.show()

## Q2 — Solve both problems independently with CVXPY

The PCW explicitly requests solving two separate optimization problems rather than using CVXPY's built-in dual-multiplier extraction.

The primal minimizes $c^Tx$ subject to $Ax\ge b$ and $x\ge0$. The dual maximizes $b^T\mu$ subject to $A^T\mu\le c$ and $\mu\ge0$.

The recorded result is **primal cost = $50, dual value = $50, duality gap approximately 0**. This illustrates strong duality for a feasible linear program with a finite optimum.

In [ ]:
# Q2 — Solve the primal and dual independently.
results = solve_concrete_primal_dual()
print("PRIMAL status:", results["primal_status"])
print("Concrete quantities:", np.round(results["x"], 6))
print("Minimum cost ($):", round(results["primal_value"], 6))
print("DUAL status:", results["dual_status"])
print("Multipliers:", np.round(results["mu"], 6))
print("Maximum dual value ($):", round(results["dual_value"], 6))
print("Duality gap:", round(results["gap"], 8))
assert abs(results["gap"]) <= 1e-3

## Q3 — Duality and the KKT conditions: Lagrangian penalty

For the simpler convex problem

$$\min_x f(x)=x^2\quad\text{subject to }x\ge1,$$

rewrite the constraint as $g(x)=1-x\le0$. Its Lagrangian is

$$\mathcal L(x,\mu)=x^2+\mu(1-x),\quad\mu\ge0.$$

The submitted PCW plots $\mu=1$, $10$, and $100$. For any **fixed** infeasible point $x<1$, the coefficient $1-x$ is positive, so increasing $\mu$ increases the Lagrangian without bound. For a fixed feasible point $x>1$, the coefficient is negative, so the maximum over $\mu\ge0$ occurs at $\mu=0$. At $x=1$, the penalty is zero for any $\mu$.

In [ ]:
# Q3 — Compare the original objective with L(x, mu) for each multiplier.
fig = plot_lagrangian_penalties(multipliers=(1, 10, 100))
plt.show()

## Q4 — Pointwise supremum and an illustrative finite step

The supremum is taken **pointwise over mu while keeping x fixed**:

$$
\sup_{\mu\ge0}\mathcal L(x,\mu)=
\begin{cases}
x^2,&x\ge1,\\
+\infty,&x<1.
\end{cases}
$$

Thus maximization over the multiplier gives the infinite-penalty objective, and minimizing that objective over x recovers the original constrained optimum.

The submitted workbook illustrates the behavior with a finite penalty:

$$
T(x)=\begin{cases}
x^2,&x\ge1,\\
20,&x<1.
\end{cases}
$$

The plot distinguishes infeasible (x<1) from feasible (x>=1) points and marks the solution (1,1). This finite-height plot is an **illustration**, not literally the same as assigning +infinity to each infeasible point. It happens to have the same minimizer for this example.

In [ ]:
# Q4 — Finite-step approximation to the infinite penalty.
fig = plot_finite_step(penalty=20)
plt.show()
print("At x=1 (feasible):", finite_step([1])[0])
print("At x=0 (infeasible):", finite_step([0])[0])

## Wrap-up — Weak and strong duality; Slater and KKT

For a minimization problem, the primal and dual optimal values can be written as

$$p^*=\inf_x\sup_{\mu\ge0}\mathcal L(x,\mu),\qquad
d^*=\sup_{\mu\ge0}\inf_x\mathcal L(x,\mu).$$

**Weak duality:** $d^*\le p^*$ (the dual is a lower bound). **Strong duality:** $d^*=p^*$ (the gap vanishes). For a convex problem in standard form, Slater's strict-feasibility condition is sufficient for strong duality when the primal optimal value is finite, under the usual domain assumptions.

In differentiable convex optimization, the four KKT conditions are **stationarity**, **primal feasibility**, **dual feasibility**, and **complementary slackness**. Slater's condition helps ensure optimal multipliers exist; it is sufficient, not necessary.

**Source boundary:** The submitted ten-page PDF only begins Question 5, asking for a KKT-based dual derivation. No answer is visible. Question 6 is not included in the supplied document. This wrap-up supplies definitions, not fictitious submitted solutions.